# Synthetic Data Augmentation with Conditional Flow Matching

## Part 3

In this section, we evaluate the impact of incorporating synthetic image data on classification performance. In particular, we examine changes in classification accuracy and $\mathsf F_1$ score in the medium-data regime, where models are trained on half of the available Fashion MNIST training set and supplemented with additional synthetic samples.

## Setup

In [1]:
!find . -mindepth 1 -exec rm -rf {} + &> /dev/null
!git clone https://github.com/ZhangLyndon/FlowMatchingAugmentation . > /dev/null 2>&1

In [2]:
!pip install -qU pip
!pip install -qU -r requirements.txt

In [3]:
import os
import sys
import argparse
import functools

# Silence tqdm output
os.environ["TQDM_DISABLE"] = "1"

# Reduce CUDA memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [4]:
import torch
import torchvision
import numpy as np

# Components for initializing an ImageNet-pretrained ResNet-18 classifier, fine-
# tuning it on Fashion MNIST, and evaluating classification performance on base-
# line, low-data, and synthetically augmented settings.
from classification import (ClassificationTrainer,
                            create_classifier, ResNetClassifier,
                            SyntheticDataGenerator, SyntheticAugmentationEvaluator,
                            create_augmented_dataset)

# Utilities for loading the Fashion MNIST dataset, computing top-k categorical
# accuracy and average cross-entropy loss, and saving training results.
from utils import get_dataloaders, AverageMeter, accuracy, save_results

import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams["font.family"] = "DejaVu Sans Mono"

We evaluate classification performance under synthetic data augmentation. 50% of the training set is combined with synthetic samples equivalent to 100% of the original dataset (6,000 per class), yielding a training set equal to 150% of the original size. Synthetic samples are generated with a guidance scale of $w = 5.0$, and performance is evaluated using both accuracy and the macro $\mathsf F_1$ score.

In [5]:
# Configure augmentation evaluation pipeline
augmentation_args = argparse.Namespace(data_root = "./data",
                                       batch_size = 16,
                                       num_workers = 0,
                                       epochs = 25,
                                       lr = 0.001,
                                       weight_decay = 1e-4,
                                       step_size = 15,
                                       gamma = 0.1,
                                       synthetic_data_dir = "./images",
                                       real_ratio = 0.5,
                                       classification_dir = "./results/classification",
                                       augmentation_dir = "./results/augmentation",
                                       checkpoint_dir = "./checkpoints",
                                       save_interval = 20,
                                       seed = 42)

# Create directory to store synthetic augmentation results
os.makedirs(augmentation_args.augmentation_dir, exist_ok = True)

# Set random seed for reproducibility
torch.manual_seed(augmentation_args.seed)
np.random.seed(augmentation_args.seed)

In [6]:
guidance_scale = 5.0
evaluator = SyntheticAugmentationEvaluator(augmentation_args, guidance_scale)
evaluator.run_low_data_experiments(augmentation_args.real_ratio, True)

Found 60000 synthetic images.
Number of epochs: 25
Number of training samples: 90000
Number of validation samples: 10000
Epoch 1/25
Training Set | Loss: 0.3211, Top-1 Accuracy: 90.65%, Top-5 Accuracy: 99.38%
Validation Set | Loss: 0.5467, Top-1 Accuracy: 81.86%, Top-5 Accuracy: 99.22%
Best Validation Loss (Up Until Now): 0.5467
_________________________________________________________________________________________________________

Epoch 2/25
Training Set | Loss: 0.1951, Top-1 Accuracy: 93.89%, Top-5 Accuracy: 99.82%
Validation Set | Loss: 0.4950, Top-1 Accuracy: 83.27%, Top-5 Accuracy: 99.55%
Best Validation Loss (Up Until Now): 0.4950
_________________________________________________________________________________________________________

Epoch 3/25
Training Set | Loss: 0.1635, Top-1 Accuracy: 94.75%, Top-5 Accuracy: 99.83%
Validation Set | Loss: 0.3759, Top-1 Accuracy: 86.50%, Top-5 Accuracy: 99.70%
Best Validation Loss (Up Until Now): 0.3759
______________________________________

Upon applying augmentation to the medium-data training split (i.e., 50% of the training set), the model achieves a classification accuracy of $90.9\%$, a macro $\mathsf F_1$ score of $0.9092$, and an optimal validation (cross-entropy) loss of $0.2698$ at epoch 19.